# Lab 02: Prompt Engineering for NLP Tasks

In this lab you will learn systematic prompt engineering techniques:
few-shot learning, chain-of-thought reasoning, and output formatting.
These skills are essential for building reliable NLP applications.

In [ ]:
# One-time setup: install the Python packages this lab uses
%pip install -q openai


In [ ]:
import os
from openai import OpenAI

client = OpenAI(
    base_url=os.environ["MODEL_ENDPOINT"],
    api_key=os.environ["MODEL_API_KEY"],
)
MODEL = os.environ["MODEL_NAME"]

## Zero-shot vs Few-shot Classification

Compare how providing examples improves the model's accuracy.

In [ ]:
test_texts = [
    "The server crashed again during peak hours",
    "Can I get a refund for my subscription?",
    "Your product is amazing, saved me hours of work!",
    "How do I reset my password?",
]

# Zero-shot
print("=== Zero-shot ===")
for text in test_texts:
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "Classify the support ticket into one of: BUG, BILLING, PRAISE, QUESTION. Reply with only the label."},
            {"role": "user", "content": text},
        ],
        max_tokens=10,
    )
    print(f"  {resp.choices[0].message.content.strip():10s} | {text}")

# Few-shot
print("\n=== Few-shot ===")
examples = [
    ("The app freezes when I click save", "BUG"),
    ("I was charged twice this month", "BILLING"),
    ("Love the new dark mode feature!", "PRAISE"),
    ("Where can I find the API docs?", "QUESTION"),
]
few_shot_msgs = [{"role": "system", "content": "Classify support tickets. Categories: BUG, BILLING, PRAISE, QUESTION."}]
for ex_text, ex_label in examples:
    few_shot_msgs.append({"role": "user", "content": ex_text})
    few_shot_msgs.append({"role": "assistant", "content": ex_label})

for text in test_texts:
    msgs = few_shot_msgs + [{"role": "user", "content": text}]
    resp = client.chat.completions.create(model=MODEL, messages=msgs, max_tokens=10)
    print(f"  {resp.choices[0].message.content.strip():10s} | {text}")

## Chain-of-Thought Reasoning

For complex tasks, asking the model to "think step by step" often
improves accuracy.

In [ ]:
ambiguous_text = "The bank was steep and covered in wildflowers."

# Direct
resp1 = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": f"What does 'bank' mean in: '{ambiguous_text}'"}],
)
print("Direct:")
print(f"  {resp1.choices[0].message.content.strip()}")

# Chain of thought
resp2 = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": f"""What does 'bank' mean in: '{ambiguous_text}'

Think step by step:
1. List possible meanings of 'bank'
2. Look at context clues in the sentence
3. Determine which meaning fits best"""}],
)
print("\nChain-of-thought:")
print(f"  {resp2.choices[0].message.content.strip()}")

## Exercise

1. Design a few-shot prompt for a classification task in your domain
2. Test with 5+ examples and measure accuracy
3. Try adding chain-of-thought -- does it help or hurt for your task?

In [ ]:
# Your code here
